# Modules over vertex algebras

A (left) *module* over a vertex (super)algebra $V$ is a vector space $M$ on which every $a \in V$ acts by a field
$$Y_M(a, z) = \sum_{n \in \mathbb Z} a_{(n)} z^{-n-1}, \qquad a_{(n)} \in \operatorname{End} M,$$
satisfying the conditions that $a_{(n)} m = 0$ for $n$ large, the vacuum acts as the identity endomorphism, and the Borcherds identity holds [Kac1998, LL2004].

This notebook shows how to build modules and compute with them. The constructions are:

| construction | what it gives | section |
|---|---|---|
| `V.adjoint_module()` | $V$ acting on itself | 1 |
| `V.simple_module(λ)`, `V.simple_modules()` | the simple modules $V_{\lambda+L}$ of a lattice vertex algebra $V_L$ | 2 |
| `A.module(matrices)` | a finite-dimensional module over a Zhu algebra $A$, a possible top level | 3 |
| `V.induced_module(U)`, `V.verma_module(h)` | Zhu's universal module with top level $U$: Verma and Weyl modules | 4 |
| `M.submodule(vectors)`, `M / N`, `M.maximal_submodule()`, `M.simple_quotient()`, `M.direct_sum(N)` | submodules, quotients, simple modules, direct sums | 5 |
| `A.verma_module(λ)` | modules with infinite-dimensional top levels | 7 |
| `Q.induced_module_up_to(U, max_weight, max_depth)` | the universal module of a quotient $Q = V/I$ | 8 |
| `M.pullback(φ)`, `M.twist(σ)`, `M.spectral_flow(h)` | new modules from old ones | 11 |

Sections 9 and 10 are about maps: morphisms of vertex algebras, and morphisms of modules with their Hom spaces, kernels and images. Section 6 grades modules by abelian groups, for example by charge or by $\mathfrak g$-weight.

**Conventions.** Elements of $V$ act on module vectors through their own methods, with the conventions of the vertex algebra (`01_getting_started.ipynb`, sections 1 and 4): `a.nproduct(m, n)` is $a_{(n)} m$, `a.ope(m)` is the list whose entry $k$ is $a_{(k)} m$, `a.no(m)` is $a_{(-1)} m$, and `a.mode(m, n)` is $a_n m$ in the physics convention, $a_n = a_{(n + \Delta_a - 1)}$. The operator `*` is not defined between an element of $V$ and a module vector. A positive-energy module has a lowest conformal weight $h$, its *top weight*, and the *depth* of a vector of weight $h + d$ is $d$.

**Prerequisites:** `01_getting_started.ipynb`, `09_zhu_algebras.ipynb`.

**Elsewhere:** fusion products, projective covers and extensions of modules are in `11_fusion_and_extensions.ipynb`; the quantum Hamiltonian reduction of modules in `08_w_algebras.ipynb`, section 9; the modules of the triplet algebra $\mathcal W(2)$ in `15_triplet_modules.ipynb`.

**Contents**

1. The category and the adjoint module
2. Lattice modules $V_{\lambda+L}$
3. Modules over Zhu algebras
4. Induced modules
5. Submodules, quotients and simple modules
6. Gradings by abelian groups
7. Infinite top levels: affine Verma modules and admissible levels
8. Modules over quotients
9. Morphisms of vertex algebras
10. Module morphisms
11. Pullbacks, twists and spectral flow
12. Limitations

## 1. The category and the adjoint module

Modules over $V$ form a category, `VertexAlgebraModules(V)`, with super and conformally graded versions; `TestSuite` checks the module axioms on its objects. As for Sage's modules over an algebra, it is contained in the category of modules over the coefficient ring. The cell below defines the universal Virasoro algebra over $\mathbb Q(c)$, prints its category of modules and the category of conformally graded super modules with basis, and checks the inclusion.

In [1]:
import vertex_algebras as va
from vertex_algebras import *

K.<c> = Frac(QQ['c'])
E.<L> = NormallyOrderedExpressions(K)
Vir = UniversalVertexAlgebra(K, E, {('L', 'L'): [L.derivative(), 2*L, 0, c/2]}, conformal_vector=L)
C = VertexAlgebraModules(Vir)
print(C)
print(C.Super().Graded().WithBasis())
print(C.Super().Graded().WithBasis().is_subcategory(Modules(K).WithBasis()))

Category of modules over Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field
Category of conformally graded super modules with basis over Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field
True


**The adjoint module.** $V$ is a module over itself, with $a_{(n)}$ acting by the $n$-th product. Its vectors are written `M(v)` for $v \in V$, and the vacuum prints as `|0>`. The cell below prints the module and its category, the OPE of $L$ with the vector ${:}LL{:}$, checks that $L_0$ acts on it by its weight 4 and that $L_{(-1)}$ acts as the normally ordered product, compares the character with that of $V$, and runs `TestSuite`.

In [2]:
M = Vir.adjoint_module()
print(M, '\n', M.category())
L = Vir.gen('L')
v = M(L.no(L))
print(L.ope(v))
print(L.mode(v, 0) == 4*v, L.no(v) == M(L.no(L.no(L))), v.degree())
print(M.character(8) == Vir.character(8))
TestSuite(M).run()

Adjoint module of Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field 
 Category of conformally graded modules with basis over Universal vertex algebra on generators ['L'] over Fraction Field of Univariate Polynomial Ring in c over Rational Field
[2*:dL L: + 1/6*d^3L, 4*:L L:, 3*dL, (c+8)*L, 0, 3*c*|0>]
True True 4
True


## 2. Lattice modules $V_{\lambda+L}$

Let $V_L$ be the lattice vertex algebra of a lattice $L$ with Gram matrix $B$ (see `05_lattice_vertex_algebras.ipynb`), and let $\lambda$ be in the *dual lattice* $L^\circ = \{\mu : B(\mu, L) \subset \mathbb Z\}$. The module $V_{\lambda+L}$ is the Fock space of the Heisenberg currents tensored with the vectors $e^p$, $p \in \lambda + L$, with
$$Y(e^q, z)\, e^p = \varepsilon(q, p - \lambda)\, z^{B(q,p)}\, E^-(q, z) E^+(q, z)\, e^{q+p},$$
where $E^\pm(q, z)$ are the exponentials of the positive and negative modes of $J_q$ [Kac1998, §5.5]. These are the simple modules of $V_L$, one for each coset in $L^\circ/L$, for any non-degenerate lattice [Don1993, DLM1997]. `V.simple_module(λ)` takes $\lambda$ in the coordinates of the basis of $L$, and uses the representative with coordinates in $[0, 1)$; `V.simple_modules()` lists them all.

**The $A_1$ lattice.** For $B = (2)$, $V_L = L_1(\mathfrak{sl}_2)$ has two simple modules. The cell below takes the conformal vector $\frac14{:}JJ{:}$ and prints the second module, the basis $e^{\pm 1/2}$ of its top level (of weight $\frac14$, the doublet of $\mathfrak{sl}_2$), and its character.

In [3]:
A1 = LatticeVertexAlgebra(QQ, matrix([[2]]))
J = A1.J(0)
V = A1.with_conformal_vector(J.no(J) / 4)
M0, M1 = V.simple_modules()
print(M1)
print(M1.weight_space(1/4).basis())
print(M1.character(5))

Module V_(λ+L), λ = (1/2), over Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field
[e^(-1/2), e^(1/2)]
2*q^(1/4) + 2*q^(5/4) + 6*q^(9/4) + 8*q^(13/4) + 14*q^(17/4) + O(q^5)


The zero modes of $e^{\alpha}$, $J$, $e^{-\alpha}$, the currents $E$, $H$, $F$ of $\mathfrak{sl}_2$, act on the doublet. The function `zero_mode_matrix` in the cell below writes the action of $a_{(0)}$ (the zero mode of a field of weight 1) on the top space as a matrix, and the cell checks the relations $[E, F] = H$ and $[H, E] = 2E$.

In [4]:
top = M1.weight_space(1/4).basis()
Ev, Hv, Fv = V.e((1,)), J, V.e((-1,))
def zero_mode_matrix(a):
    return matrix(QQ, [[a.nproduct(u, 0).monomial_coefficients().get(w.leading_support(), 0)
                        for u in top] for w in top])
Em, Hm, Fm = [zero_mode_matrix(a) for a in (Ev, Hv, Fv)]
print(Em, '\n'); print(Hm, '\n'); print(Fm)
print(Em*Fm - Fm*Em == Hm, Hm*Em - Em*Hm == 2*Em)

[0 0]
[1 0] 

[-1  0]
[ 0  1] 

[0 1]
[0 0]
True True


**The $A_2$ lattice** has three simple modules, of lowest conformal weights $0$, $\frac13$, $\frac13$, and with top space dimensions $1$, $3$, $3$ (the trivial, the standard and the dual representation of $\mathfrak{sl}_3$). For each module the cell below prints the coset representative $\lambda$, the dimensions of the weight spaces of depth $0, \dots, 3$ and the basis of the top space; `highest_weight_vector()` returns a vector of lowest conformal weight. The last line shows that these are the top spaces that the Zhu algebra of the lattice finds (`09_zhu_algebras.ipynb`, section 8).

In [5]:
A2 = LatticeVertexAlgebra(QQ, matrix([[2, -1], [-1, 2]]))
J0, J1 = A2.heisenberg_currents()
VA2 = A2.with_conformal_vector((2*J0.no(J0) + J0.no(J1) + J1.no(J0) + 2*J1.no(J1)) / 6)
for N in VA2.simple_modules():
    lam = N.coset_representative()
    h = N.highest_weight_vector().degree()
    print(lam, [N.weight_space(h + d).dimension() for d in range(4)], N.weight_space(h).basis())
print(VA2.zhu_algebra().top_levels())

(0, 0) [1, 8, 17, 46] [|0>]
(1/3, 2/3) [3, 9, 27, 57] [e^(-2/3,-1/3), e^(1/3,-1/3), e^(1/3,2/3)]
(2/3, 1/3) [3, 9, 27, 57] [e^(-1/3,-2/3), e^(-1/3,1/3), e^(2/3,1/3)]
[(0, [(0, 0)]), (1/3, [(-2/3, -1/3), (1/3, -1/3), (1/3, 2/3)]), (1/3, [(-1/3, -2/3), (-1/3, 1/3), (2/3, 1/3)])]


**Odd lattices** give vertex superalgebras and super modules. For $\sqrt 3\,\mathbb Z$ ($B = (3)$, the $N = 2$ superconformal algebra at $c = 1$) there are three modules, of lowest conformal weights $0$, $\frac16$, $\frac16$; the vector $e^{\lambda + q}$ has parity $B(q, q) \bmod 2$. The cell below prints, for each module, $\lambda$, the highest-weight vector of the module, its weight and the character, then the category and an odd vector.

In [6]:
L3 = LatticeVertexAlgebra(QQ, matrix([[3]]))
J3 = L3.J(0)
V3 = L3.with_conformal_vector(J3.no(J3) / 6)
for N in V3.simple_modules():
    v = N.highest_weight_vector()
    print(N.coset_representative(), v, v.degree(), N.character(3))
print(N.category())
N = V3.simple_module([1/3])
print(N.weight_space(2/3).basis(), N.monomial(((), (-2/3,))).parity())

(0,) |0> 0 1 + q + 2*q^(3/2) + 2*q^2 + 2*q^(5/2) + O(q^3)
(1/3,) e^(1/3) 1/6 q^(1/6) + q^(2/3) + q^(7/6) + q^(5/3) + 2*q^(13/6) + 3*q^(8/3) + O(q^3)
(2/3,) e^(-1/3) 1/6 q^(1/6) + q^(2/3) + q^(7/6) + q^(5/3) + 2*q^(13/6) + 3*q^(8/3) + O(q^3)
Category of conformally graded super modules with basis over Lattice vertex algebra of rank 1 with Gram matrix [3] over Rational Field
[e^(-2/3)] 1


**A background charge** changes the conformal weights to $\Delta(p) = \frac12 B(p, p) + B(\beta, p)$. Consider $T = \frac14{:}JJ{:} + \frac12 \partial J$ acting on $A_1$, of central charge $-5$; with respect to this grading, the vacuum module now has two vectors of conformal weight 0:

In [7]:
Vb = A1.with_conformal_vector(J.no(J) / 4 + J.derivative() / 2)
print(Vb.central_charge())
for N in Vb.simple_modules():
    print(N.coset_representative(), N.character(3))

-5
(0,) 2 + 2*q + 6*q^2 + O(q^3)
(1/2,) q^(-1/4) + 3*q^(3/4) + 4*q^(7/4) + 7*q^(11/4) + O(q^3)


**Indefinite lattices** work as well. Their weight spaces are infinite-dimensional, but with the lattice grading of $V_L$ (`with_horizontal_grading('lattice')`, `05_lattice_vertex_algebras.ipynb`, section 5) the graded pieces are finite. The lattice with $B = \begin{pmatrix} 0 & 2 \\ 2 & 0 \end{pmatrix}$ has four simple modules, and $e^p$ has grade $p$. In $V_{(0, 1/2) + L}$ the vector $e^{(0,1/2)}$ has weight $\frac12 B(p, p) = 0$, and the piece of grade $(0, \frac12)$ is the Fock space of two bosons on it, with dimensions $1, 2, 5, 10, 20, \dots$:

In [8]:
H2 = LatticeVertexAlgebra(QQ, matrix([[0, 2], [2, 0]]))
K0, K1 = H2.heisenberg_currents()
Hg = H2.with_conformal_vector((K0.no(K1) + K1.no(K0)) / 4).with_horizontal_grading('lattice')
Ms = Hg.simple_modules()
print([N.coset_representative() for N in Ms])
N = Ms[1]
print(N.weight_space(1).is_finite_dimensional(), [N.weight_space(w, horizontal_grade=(0, 1/2)).dimension() for w in range(5)])

[(0, 0), (0, 1/2), (1/2, 0), (1/2, 1/2)]
False [1, 2, 5, 10, 20]


## 3. Modules over Zhu algebras

By Zhu's correspondence [Zhu1996] the top space of a positive-energy module of $V$ is a module over the Zhu algebra $A(V)$ (see `09_zhu_algebras.ipynb`), and every module of $A(V)$ is the top space of some positive-energy module (section 4). `A.module(matrices)` builds a finite-dimensional module over a Zhu algebra $A$ from the matrices of its generators, given as a dict over the names of the generators of $V$. The matrices are checked against the defining relations of $A$, and, for the Zhu algebra of a quotient, against its ideal; a `ValueError` names the first relation that fails.

**Virasoro.** $A = \mathbb C[x]$, $x = [L]$, has a one-dimensional module for every value $h$ of $x$. The Zhu algebra of the Lee–Yang model (`09_zhu_algebras.ipynb`, section 6) only has the modules $h = 0$ and $h = -\frac15$. The cell below builds the module $h = -\frac15$ and shows that $h = 1$ is refused.

In [9]:
EQ.<Lq> = NormallyOrderedExpressions(QQ)
VirLY = UniversalVertexAlgebra(QQ, EQ, {('Lq', 'Lq'): [Lq.derivative(), 2*Lq, 0, -11/5]}, conformal_vector=Lq)
Lq = VirLY.gen('Lq')
Z = VirLY.quotient([Lq.no(Lq) - 3/10*Lq.derivative(2)]).zhu_algebra_up_to(4)
print(Z.module({'Lq': -1/5}))
try:
    Z.module({'Lq': 1})
except ValueError as err:
    print('ValueError:', err)

Module of dimension 1 over Quotient of Zhu algebra of Universal vertex algebra on generators ['Lq'] over Rational Field by the images of the ideal of weight <= 4: an algebra mapping onto the Zhu algebra of Quotient of Universal vertex algebra on generators ['Lq'] over Rational Field by the ideal generated by [:Lq Lq: - 3/10*d^2Lq]
ValueError: the matrices do not satisfy the relation [Lq] + 5*[Lq]^2 = 0 of the Zhu algebra


**Affine $\mathfrak{sl}_2$.** $A = U(\mathfrak{sl}_2)$ [FZ1992], and the doublet is a module. The image of the Sugawara vector acts on it by $\frac{3}{4(k+2)} = \frac14$ at level $k = 1$. The Zhu algebra of $L_1(\mathfrak{sl}_2)$ has the doublet as a representation but not the triplet. So of the irreducible $\mathfrak{sl}_2$-modules, only those of dimension 1 and 2 are top spaces of $L_1(\mathfrak{sl}_2)$-modules.

In [10]:
VA = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=1, names=['E', 'H', 'F'], conformal_vector='sugawara')
AA = VA.zhu_algebra()
doublet = {'E': matrix([[0, 1], [0, 0]]), 'H': matrix([[1, 0], [0, -1]]), 'F': matrix([[0, 0], [1, 0]])}
triplet = {'E': matrix([[0, -2, 0], [0, 0, 1], [0, 0, 0]]), 'H': matrix([[2, 0, 0], [0, 0, 0], [0, 0, -2]]),
           'F': matrix([[0, 0, 0], [-1, 0, 0], [0, 2, 0]])}
U2 = AA.module(doublet)
print(U2, '\n', U2.representation_matrix(AA(VA.conformal_vector())))
E1 = VA.gen('E')
ZL = VA.quotient([E1.no(E1)]).zhu_algebra_up_to(3)
print(ZL.module(doublet).dimension())
try:
    ZL.module(triplet)
except ValueError as err:
    print('ValueError:', err)

Module of dimension 2 over Zhu algebra of Universal affine vertex algebra of type ['A', 1] at level 1 over Rational Field 
 [1/4   0]
[  0 1/4]
2
ValueError: the matrices do not satisfy the relation [F]^2 = 0 of the Zhu algebra


**Super modules.** The $bc$ system with weights $1, 0$ has the Clifford algebra as its Zhu algebra (`09_zhu_algebras.ipynb`, section 4), with the module $\mathbb C^{1|1}$ (the parities of the basis vectors are given with `parities=`; since $c$ has weight 0, the library only claims an algebra mapping onto the Zhu algebra).

The H-twisted (Ramond) Zhu algebra of the $N = 1$ superconformal algebra has the relation $[G]^2 = [L] - c/24$ (`09_zhu_algebras.ipynb`, section 10). In the Ramond sector of the tricritical Ising model ($c = \frac{7}{10}$), the quotient additionally imposes the relation $1280[L]^2 - 608[L] + 21 = 0$, which leaves the modules of dimension $1|1$ at $h = \frac{3}{80}$ and $h = \frac{7}{16}$. The cell below finds the singular vector of weight 4 (killed by $L_1, L_2, G_{1/2}, G_{3/2}$), builds the H-twisted Zhu algebra of the quotient, and tries the modules `ramond(h)` for three values of $h$.

In [11]:
X.<b, cg> = NormallyOrderedExpressions(QQ, parities={'b': 1, 'cg': 1})
BC = UniversalVertexAlgebra(QQ, X, {('b', 'cg'): [1]}, conformal_vector=-b*cg.derivative())
Ubc = BC.zhu_algebra().module({'b': matrix([[0, 1], [0, 0]]), 'cg': matrix([[0, 0], [1, 0]])}, parities=[0, 1])
print(Ubc, '\n', Ubc.category())

NT.<Lt, Gt> = NormallyOrderedExpressions(QQ, parities={'Gt': 1})
TI = UniversalVertexAlgebra(QQ, NT, {('Lt', 'Lt'): [Lt.derivative(), 2*Lt, 0, 7/20],
                                     ('Lt', 'Gt'): [Gt.derivative(), 3/2*Gt],
                                     ('Gt', 'Gt'): [2*Lt, 0, 7/15]}, conformal_vector=Lt)
Ltv, Gtv = TI.gens()
S4 = TI.weight_space(4).basis()
rows = []
for g, n in [(Ltv, 2), (Ltv, 3), (Gtv, 1), (Gtv, 2)]:          # killed by L_1, L_2, G_{1/2}, G_{3/2}
    imgs = [g.nproduct(s, n).monomial_coefficients() for s in S4]
    ks = sorted(set(kk for d in imgs for kk in d))
    rows += [[d.get(kk, 0) for d in imgs] for kk in ks]
[sv] = [sum(cf*s for cf, s in zip(r, S4)) for r in matrix(QQ, rows).right_kernel_matrix().rows()]
ZR = TI.quotient([sv]).zhu_algebra_up_to(6, twist='ramond')
def ramond(h):
    return {'Lt': matrix([[h, 0], [0, h]]), 'Gt': matrix([[0, 1], [h - 7/240, 0]])}
for h in (3/80, 7/16, 1):
    try:
        ZR.module(ramond(h), parities=[0, 1])
        print(h, 'a module')
    except ValueError as err:
        print(h, 'ValueError:', err)

Module of dimension 2 over Algebra mapping onto the Zhu algebra of Universal vertex algebra on generators ['b', 'cg'] over Rational Field 
 Category of finite dimensional super modules with basis over Rational Field
3/80 a module
7/16 a module
1 ValueError: the matrices do not satisfy the relation 21 - 608*[Lt] + 1280*[Lt]^2 = 0 of the Zhu algebra


## 4. Induced modules

For a universal vertex algebra $V$ with generators of positive weight and a module $U$ over its Zhu algebra, `V.induced_module(U)` is Zhu's universal positive-energy module $\bar M(U)$ with top level $U$ [Zhu1996]: every positive-energy module generated by a top level isomorphic to $U$ is a quotient of it. Concretely:

* its basis is the ordered monomials of the negative modes of the strong generators applied to a basis of $U$ (a PBW basis);
* positive modes kill $U$, and zero modes act on $U$ by the matrices of $U$;
* modes are reordered by the commutator formula $[a_m, b_n] = \sum_j \binom{m + \Delta_a - 1}{j} (a_{(j)} b)_{m+n}$.

`V.verma_module(h)` is the case of a one-dimensional $U$, with $h$ the value of the generators (a dict, or a number for a single generator).

**The Virasoro Verma module $M(c, h)$**, with $c$ and $h$ symbolic: the cell below prints the module, the vector $w = L_{-1}^2 v$, the actions of $L_1$, $L_2$ and $L_0$ on it, the dimensions of the weight spaces (the numbers of partitions) and the basis of depth 2.

In [12]:
Kch.<c, h> = Frac(QQ['c', 'h'])
Eh.<T> = NormallyOrderedExpressions(Kch)
VirH = UniversalVertexAlgebra(Kch, Eh, {('T', 'T'): [T.derivative(), 2*T, 0, c/2]}, conformal_vector=T)
T = VirH.gen('T')
Mh = VirH.verma_module(h)
v = Mh.highest_weight_vector()
w = T.mode(T.mode(v, -1), -1)
print(Mh, '\n', w)
print(T.mode(w, 1), '|', T.mode(w, 2), '|', T.mode(w, 0) == (h + 2)*w)
print([Mh.weight_space(h + d).dimension() for d in range(8)])
print(Mh.weight_space(h + 2).basis())

Verma module of highest weight {'T': h} over Universal vertex algebra on generators ['T'] over Fraction Field of Multivariate Polynomial Ring in c, h over Rational Field 
 T_{-1} T_{-1} v
(4*h+2)*T_{-1} v | 6*h*v | True
[1, 1, 2, 3, 5, 7, 11, 15]
[T_{-2} v, T_{-1} T_{-1} v]


**Singular vectors at depth 2.** A vector of weight $h + 2$ is singular (killed by all $L_n$, $n \ge 1$) if and only if $L_1$ and $L_2$ kill it. So a singular vector exists at depth 2 if and only if the matrix of $(L_1, L_2)$, from the depth-2 space to the depth-1 and depth-0 spaces, is singular. The cell below computes this matrix and its determinant, $-(16h^2 + 2(c - 5)h + c)$, which vanishes if and only if $c = 2h(5 - 8h)/(2h + 1)$. This is a factor of the Kac determinant [Kac1979]; the remaining factor $h$ belongs to the singular vector $L_{-1} v$ at depth 1, which exists if and only if $h = 0$.

In [13]:
S2 = Mh.weight_space(h + 2).basis()                      # L_{-2} v, L_{-1}^2 v
k1 = Mh.weight_space(h + 1).basis_keys()[0]              # L_{-1} v
k0 = Mh.weight_space(h).basis_keys()[0]                  # v
mat = matrix(Kch, [[T.mode(s, 1).monomial_coefficients().get(k1, 0),
                    T.mode(s, 2).monomial_coefficients().get(k0, 0)] for s in S2])
print(S2); print(mat)
print(factor(mat.det()))

[T_{-2} v, T_{-1} T_{-1} v]
[          3 1/2*c + 4*h]
[    4*h + 2         6*h]
(-1) * (2*c*h + 16*h^2 + c - 10*h)


At $c = h = \frac12$ (the module $\varepsilon$ of the Ising model [BPZ1984]) the singular vector is $(L_{-2} - \frac34 L_{-1}^2)v$; at $h = 0$ it is $L_{-1} v$. The cell below checks both and prints the character of the Verma module $M(\frac12, \frac12)$.

In [14]:
Eq.<Ti> = NormallyOrderedExpressions(QQ)
Ising = UniversalVertexAlgebra(QQ, Eq, {('Ti', 'Ti'): [Ti.derivative(), 2*Ti, 0, 1/4]}, conformal_vector=Ti)
Ti = Ising.gen('Ti')
M12 = Ising.verma_module(1/2)
v = M12.highest_weight_vector()
s = Ti.mode(v, -2) - 3/4 * Ti.mode(Ti.mode(v, -1), -1)
print(s, '|', Ti.mode(s, 1), Ti.mode(s, 2))
M0 = Ising.verma_module(0)
print(Ti.mode(Ti.mode(M0.highest_weight_vector(), -1), 1))
print(M12.character(5))

Ti_{-2} v - 3/4*Ti_{-1} Ti_{-1} v | 0 0
0
q^(1/2) + q^(3/2) + 2*q^(5/2) + 3*q^(7/2) + 5*q^(9/2) + O(q^5)


**Affine $\mathfrak{sl}_2$ at level $k$.** The Weyl module of the doublet has top weight $\frac{3}{4(k+2)}$ (the eigenvalue of the Sugawara vector on the doublet). Its vector $E_{-1} u_0$ satisfies $F_1 E_{-1} u_0 = (k - 1) u_0$, and $H_1$, $E_1$ kill it, so it is singular exactly at level 1, where the doublet is an integrable top. The cell below prints the top weight, the dimensions of depth $0, \dots, 3$, and the actions of the three positive modes on $E_{-1} u_0$.

In [15]:
Kk.<k> = Frac(QQ['k'])
Vk = UniversalAffineVertexAlgebra(Kk, ['A', 1], level=k, names=['E', 'H', 'F'], conformal_vector='sugawara')
W = Vk.induced_module(Vk.zhu_algebra().module(doublet))
print(W.top_weight(), [W.weight_space(W.top_weight() + d).dimension() for d in range(4)])
Ek, Hk, Fk = Vk.gens()
u0 = W.highest_weight_vector()
x = Ek.mode(u0, -1)
print(x, '|', Fk.mode(x, 1), '|', Hk.mode(x, 1), Ek.mode(x, 1))

3/4/(k + 2) [2, 6, 18, 44]
E_{-1} u[0] | (k-1)*u[0] | 0 0


**Zhu's theorem on the top space.** On the top space, the zero modes $o(a) = a_{(\Delta_a - 1)}$ act by the matrices of $U$ of the images $[a]$ in the Zhu algebra. The cell below checks this for every basis vector $a$ of $V^k(\mathfrak{sl}_2)$ of weight 2 (nine of them), comparing $a_{(1)}$ on the two top vectors with the matrix of $[a]$ on $U$.

In [16]:
Ak = Vk.zhu_algebra()
U = W.top_level()
ok = True
for key in Vk.weight_space(2).basis_keys():
    a = Vk.monomial(key)
    m = U.representation_matrix(Ak(a))
    for j in range(2):
        img = a.nproduct(W.monomial(((), j)), 1)
        ok = ok and dict(img.monomial_coefficients()) == {((), r): m[r, j] for r in range(2) if m[r, j]}
print(ok, len(Vk.weight_space(2).basis_keys()))

True 9


**The trivial top.** $V^k(\mathfrak{sl}_2)$ itself is the module induced from the trivial module of $U(\mathfrak{sl}_2)$, by $a \mapsto a_{(-1)} v$. The cell below compares the dimensions and checks that $a \mapsto a_{(-1)}v$ is compatible with the products $a_{(n)}$ for one pair of elements.

In [17]:
Mt = Vk.induced_module(Ak.module({'E': 0, 'H': 0, 'F': 0}))
vt = Mt.highest_weight_vector()
print([Mt.weight_space(d).dimension() for d in range(4)], [Vk.weight_space(d).dimension() for d in range(4)])
a, bb = Ek, Fk.no(Hk)
print(all(a.nproduct(bb, n).nproduct(vt, -1) == a.nproduct(bb.no(vt), n) for n in range(-2, 3)))

[1, 3, 9, 22] [1, 3, 9, 22]
True


**Non-linear OPEs: the $W_3$ algebra** [Zam1985]. Here the reordering of modes involves the composite field $\Lambda = {:}LL{:} - \frac3{10}\partial^2 L$, and the result is a module only if the rewriting is consistent, that is, if $V$ satisfies the Jacobi identity. The function `failures` below counts, on all vectors of depth $\le 2$ of a Verma module, the failures of the commutator formula for the pairs $(W, W)$ and $(W, L)$. With Zamolodchikov's $\beta = 16/(22 + 5c)$ there are none; with a wrong $\beta$ there are some. (This $W_3$ is given by conformal weights, without a conformal vector, so the module is graded by depth, with top weight $0$; $L_0$ acts on the top by the chosen value $c + 1$.)

In [18]:
K.<c> = Frac(QQ['c'])                                    # c was rebound to Frac(QQ['c', 'h']) above
E3.<L3, W3> = NormallyOrderedExpressions(K)
def w3(beta):
    Lam = L3*L3 - 3/10*L3.derivative(2)
    return UniversalVertexAlgebra(K, E3, {('L3', 'L3'): [L3.derivative(), 2*L3, 0, c/2],
        ('L3', 'W3'): [W3.derivative(), 3*W3],
        ('W3', 'W3'): [beta*Lam.derivative() + L3.derivative(3)/15, 2*beta*Lam + 3/10*L3.derivative(2),
                       L3.derivative(), 2*L3, 0, c/3]}, degrees={'L3': 2, 'W3': 3})

def failures(V):
    M = V.verma_module({'L3': c + 1, 'W3': 1/c})
    Lg, Wg = V.gens()
    bad = total = 0
    for x, y in [(Wg, Wg), (Wg, Lg)]:
        xy = x.ope(y)
        for m in [M.monomial(kk) for d in (0, 1, 2) for kk in M._depth_keys(d)]:
            for q in range(-1, 3):
                for p in range(4):
                    lhs = x.nproduct(y.nproduct(m, q), p) - y.nproduct(x.nproduct(m, p), q)
                    rhs = sum((binomial(p, j) * xy[j].nproduct(m, p + q - j)
                               for j in range(min(p + 1, len(xy)))), M.zero())
                    total += 1
                    bad += lhs != rhs
    return bad, total

print('Zamolodchikov:', failures(w3(16/(22 + 5*c))), '  wrong beta:', failures(w3(1)))
MW = w3(16/(22 + 5*c)).verma_module({'L3': c + 1, 'W3': 1/c})
print(MW.top_weight(), [MW.weight_space(d).dimension() for d in range(6)])   # weights only, no T: top weight 0
TestSuite(MW).run()

Zamolodchikov: (0, 256)   wrong beta: (9, 256)
0 [1, 2, 5, 10, 20, 36]


## 5. Submodules, quotients and simple modules

The constructions:

* `M.submodule(vectors)` is the submodule *generated* by the vectors: all $a_{(n)} v$, applied repeatedly;
* `M / N` is the quotient by a submodule $N$; its vectors are normal forms;
* `M.maximal_submodule()` is the maximal proper submodule, the largest graded submodule that meets the top space trivially;
* `M.simple_quotient()` is `M / M.maximal_submodule()`, Zhu's simple module $L(U)$ for an induced module $\bar M(U)$.

Weight spaces are computed exactly. A submodule is spanned by raising words in the modes of the strong generators applied to the closure of its generators under the non-raising modes. The maximal submodule is computed depth by depth, as the vectors that every lowering mode sends into it. Submodules are modules themselves, with canonical bases of their weight spaces (reduced echelon forms, with leading monomials as pivots).

**The Ising model.** The three simple quotients $L(\frac12, h)$, $h = 0, \frac12, \frac1{16}$, of Verma modules are the three modules of the Ising model [BPZ1984]. The cell below prints the dimensions of their weight spaces up to depth 10, and the character of $L(\frac12, \frac1{16})$, which is $q^{1/16}\prod_{n \ge 1}(1 + q^n)$.

In [19]:
for h in (0, 1/2, 1/16):
    S = Ising.verma_module(h).simple_quotient()
    print(h, [S.weight_space(h + d).dimension() for d in range(11)])
print(S, '\n', S.character(6))

0 [1, 0, 1, 1, 2, 2, 3, 3, 5, 5, 7]
1/2 [1, 1, 1, 1, 2, 2, 3, 4, 5, 6, 8]
1/16 [1, 1, 1, 2, 2, 3, 4, 5, 6, 8, 10]
Simple quotient of Verma module of highest weight {'Ti': 1/16} over Universal vertex algebra on generators ['Ti'] over Rational Field 
 q^(1/16) + q^(17/16) + q^(33/16) + 2*q^(49/16) + 2*q^(65/16) + 3*q^(81/16) + O(q^6)


**Submodules of $M(\frac12, \frac12)$.** The singular vector $s = (L_{-2} - \frac34 L_{-1}^2) v$ generates a copy of the Verma module $M(\frac12, \frac52)$: the dimensions of the submodule $N$ it generates are those of a Verma module two levels up. `N.express(x)` writes a vector of the submodule as a descendant of $s$, here in the products $T_{(n)}$ ($T_{(0)} = L_{-1}$, $T_{(-2)} = L_{-3}$). The maximal submodule $J$ also contains a singular vector $s_3$ of depth 3, which the cell finds as the common kernel of $L_1$ and $L_2$ on the depth-3 space; $J$ is the sum of the submodules generated by $s$ and $s_3$, and `<=` tests inclusion of submodules.

In [20]:
M = Ising.verma_module(1/2); v = M.highest_weight_vector()
s = Ti.mode(v, -2) - 3/4 * Ti.mode(Ti.mode(v, -1), -1)
N = M.submodule([s], names=['s'])
print(N, '\n', [N.weight_space(1/2 + d).dimension() for d in range(9)])
x = Ti.mode(Ti.mode(s, -1), -1) + 2 * Ti.mode(s, -3)
print(x in N, v in N, '|', N.express(x))
J = M.maximal_submodule()
print(J.weight_space(1/2 + 2).basis(), [J.weight_space(1/2 + d).dimension() for d in range(9)])
Y = M.weight_space(1/2 + 3).basis()                     # the singular vectors of depth 3
keys = sorted({k for y in Y for m in (1, 2) for k in Ti.mode(y, m).monomial_coefficients()})
A = matrix(QQ, [[Ti.mode(y, m).monomial_coefficients().get(k, 0) for m in (1, 2) for k in keys] for y in Y])
s3 = sum(cf * y for cf, y in zip(A.left_kernel().basis()[0], Y)); print(s3)
N3 = M.submodule([s3], names=['s3'])
print([(N + N3).weight_space(1/2 + d).dimension() for d in range(9)], N3 <= N, N <= N + N3)

Submodule of Verma module of highest weight {'Ti': 1/2} over Universal vertex algebra on generators ['Ti'] over Rational Field generated by [Ti_{-2} v - 3/4*Ti_{-1} Ti_{-1} v] 
 [0, 0, 1, 1, 2, 3, 5, 7, 11]
True False | Ti_(0) Ti_(0) s + 2*Ti_(-2) s
[-4/3*Ti_{-2} v + Ti_{-1} Ti_{-1} v] [0, 0, 1, 2, 3, 5, 8, 11, 17]
Ti_{-3} v - 4*Ti_{-2} Ti_{-1} v + 4/3*Ti_{-1} Ti_{-1} Ti_{-1} v
[0, 0, 1, 2, 3, 5, 8, 11, 17] False True


**Lee–Yang.** The simple quotient of the adjoint module of the universal Virasoro algebra at $c = -\frac{22}{5}$ is the quotient by the maximal ideal, generated by $\Lambda$; the cell below compares it with the quotient vertex algebra of `06_subalgebras_ideals_quotients.ipynb`, section 5. The Verma module $M(-\frac{22}{5}, -\frac15)$ has as simple quotient the second module of the Lee–Yang model, whose character is the other Rogers–Ramanujan series; in it $L_{-1}^2 v = \frac25 L_{-2} v$.

In [21]:
LYa = VirLY.adjoint_module().simple_quotient()
print([LYa.weight_space(w).dimension() for w in range(11)])
print([VirLY.quotient([Lq.no(Lq) - 3/10*Lq.derivative(2)]).weight_space(w).dimension() for w in range(11)])
S = VirLY.verma_module(-1/5).simple_quotient(); v = S.highest_weight_vector()
print([S.weight_space(-1/5 + d).dimension() for d in range(11)], Lq.mode(Lq.mode(v, -1), -1))

[1, 0, 1, 1, 1, 1, 2, 2, 3, 3, 4]
[1, 0, 1, 1, 1, 1, 2, 2, 3, 3, 4]


[1, 1, 1, 1, 2, 2, 3, 3, 4, 5, 6] 2/5*Lq_{-2} v


**Affine $\mathfrak{sl}_2$ at level 1.** The simple quotients of the Weyl modules of the trivial module and of the doublet are the two lattice modules of $A_1$ of section 2 (by the Frenkel–Kac construction, section 9), as the dimensions show; at level 2 the vacuum gives $L_2(\mathfrak{sl}_2)$:

In [22]:
for U, LM in zip((AA.module({'E': 0, 'H': 0, 'F': 0}), AA.module(doublet)), V.simple_modules()):
    L = VA.induced_module(U).simple_quotient()
    h = L.top_weight()
    print(h, [L.weight_space(h + d).dimension() for d in range(6)], [LM.weight_space(h + d).dimension() for d in range(6)])
V2 = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=2, names=['E', 'H', 'F'], conformal_vector='sugawara')
L2 = V2.induced_module(V2.zhu_algebra().module({'E': 0, 'H': 0, 'F': 0})).simple_quotient()
print([L2.weight_space(d).dimension() for d in range(6)])

0 [1, 3, 4, 7, 13, 19] [1, 3, 4, 7, 13, 19]


1/4 [2, 2, 6, 8, 14, 20] [2, 2, 6, 8, 14, 20]
[1, 3, 9, 15, 30, 54]


**Direct sums** of modules, `M.direct_sum(N)`, with `embed(i, x)` putting a vector in the $i$-th summand; and a quotient of $M \oplus M$ by a diagonal submodule, which is again $M$. Lattice modules are simple: `simple_quotient()` returns the module itself. The cell ends with `TestSuite` on a quotient and on the maximal submodule $J$ above.

In [23]:
Ma, Mb = Ising.verma_module(0), Ising.verma_module(1/16)
D = Ma.direct_sum(Mb)
x = D.embed(0, Ma.highest_weight_vector()) + D.embed(1, Mb.highest_weight_vector())
print(Ti.mode(x, -1), '|', D.character(3))
DD = Ma.direct_sum(Ma)
diag = DD.submodule([DD.embed(0, Ma.highest_weight_vector()) + DD.embed(1, Ma.highest_weight_vector())])
print((DD / diag).character(5) == Ma.character(5))
print(M1.simple_quotient() is M1, M1.maximal_submodule().weight_space(1/4).dimension())
TestSuite(DD / diag).run(); TestSuite(J).run()

[0] Ti_{-1} v + [1] Ti_{-1} v | 1 + q^(1/16) + q + q^(17/16) + 2*q^2 + 2*q^(33/16) + O(q^3)
True
True 0


## 6. Gradings by abelian groups

Besides the conformal weight, a vertex algebra can be graded by an abelian group $A$ (a *horizontal grading*, for example by a charge or by a root lattice). A module $M$ is then graded by a group $B$ containing $A$, with an embedding $\iota: A \to B$ (`M.horizontal_grading_embedding()`), such that $a_{(n)} m$ has grade $\iota(\mathrm{grade}\, a) + \mathrm{grade}\, m$. The grades of an indecomposable module lie in a single coset of $\iota(A)$, so $B$ is in general larger than $A$. The methods are `M.horizontal_grading_group()` and `M.horizontal_grade(v)`, and `weight_space` and `character` take a `horizontal_grade` in $B$.

**Lattice modules.** With the lattice grading of $V_L$, the grades of the modules lie in the dual lattice $L^\circ$, with $e^p$ of grade $p$. (For another grading of $V_L$, they lie in the pushout of $A \leftarrow L \rightarrow L^\circ$.) All simple modules share the group $B$, so their direct sums are graded. The cell below prints $B$ and a grade for $A_1$, the dimensions of grade $\frac12$ (the numbers of partitions), a graded character of a direct sum, and, for $A_2$ graded by $\mathbb Z/3$, the group $B$, of order 9.

In [24]:
Vg = V.with_horizontal_grading('lattice')
N0, N1 = Vg.simple_modules()
print(N1.horizontal_grading_group(), '|', N1.horizontal_grade(N1.highest_weight_vector()), N1.horizontal_grading_embedding()(1))
print([N1.weight_space(1/4 + d, horizontal_grade=(1/2,)).dimension() for d in range(6)])   # partitions
print(N0.direct_sum(N1).character(3, horizontal_grade=(-1/2,)))
A2g = A2.with_horizontal_grading([1, 2], group=Zmod(3))
B3 = A2g.simple_modules()[1].horizontal_grading_group(); print(B3, B3.cardinality())

Free module of degree 1 and rank 1 over Integer Ring
Echelon basis matrix:
[1/2] | (-1/2) (1)
[1, 1, 2, 3, 5, 7]
q^(1/4) + q^(5/4) + 2*q^(9/4) + O(q^3)


Finitely generated module V/W over Integer Ring with invariants (9) 9


**Affine vertex algebras** are graded by the root lattice $Q$; their modules by the weights, $P \otimes R$ for the coefficient ring $R$ (in fundamental weight coordinates), which contains $Q$ through the Cartan matrix. The grades of a module over the Zhu algebra are by default the eigenvalues of the zero modes of the Cartan currents (or given with `horizontal_grades=`), and an induced module grades its PBW monomials accordingly. The cell below shows the Weyl module of the doublet over $\mathbb Q(k)$ and, at level 1, its simple quotient, graded like the lattice module $V_{1/2 + A_1}$ ($J_{(0)} e^p = 2p\, e^p$): rows are depths $0, \dots, 3$, columns the $\mathfrak{sl}_2$-weights $-5, -3, \dots, 5$.

In [25]:
U = W.top_level(); print(U.horizontal_grades(), U.horizontal_grading_group())
h = W.top_weight()
print([[W.weight_space(h + d, horizontal_grade=(m,)).dimension() for m in range(-5, 6, 2)] for d in range(4)])
L1 = VA.induced_module(AA.module(doublet)).simple_quotient()
print([[L1.weight_space(1/4 + d, horizontal_grade=(m,)).dimension() for m in range(-5, 6, 2)] for d in range(4)])
print([[N1.weight_space(1/4 + d, horizontal_grade=(m/2,)).dimension() for m in range(-5, 6, 2)] for d in range(4)])

[(1), (-1)] Vector space of dimension 1 over Fraction Field of Univariate Polynomial Ring in k over Rational Field
[[0, 0, 1, 1, 0, 0], [0, 1, 2, 2, 1, 0], [1, 3, 5, 5, 3, 1], [3, 7, 11, 11, 7, 3]]
[[0, 0, 1, 1, 0, 0], [0, 0, 1, 1, 0, 0], [0, 1, 2, 2, 1, 0], [0, 1, 3, 3, 1, 0]]
[[0, 0, 1, 1, 0, 0], [0, 0, 1, 1, 0, 0], [0, 1, 2, 2, 1, 0], [0, 1, 3, 3, 1, 0]]


**Grades outside $A$.** The $N = 2$ superconformal algebra is graded by the charge of $J$, with $G^\pm$ of charge $\pm 1$, so $A = \mathbb Z$. Its Verma module of charge $\frac13$ is graded by $\mathbb Q \supset \mathbb Z$. The cell below prints the grading group, the grade of the highest weight vector, the dimensions in charge $\frac13 - 1$, and the character in charge $\frac13$.

In [26]:
X2.<Ln, Jn, Gp, Gm> = NormallyOrderedExpressions(QQ, parities={'Gp': 1, 'Gm': 1})
N2 = UniversalVertexAlgebra(QQ, X2, {
    ('Ln', 'Ln'): [Ln.derivative(), 2*Ln, 0, 1/2], ('Ln', 'Jn'): [Jn.derivative(), Jn],
    ('Ln', 'Gp'): [Gp.derivative(), 3/2*Gp], ('Ln', 'Gm'): [Gm.derivative(), 3/2*Gm],
    ('Jn', 'Jn'): [0, 1/3], ('Jn', 'Gp'): [Gp], ('Jn', 'Gm'): [-Gm],
    ('Gp', 'Gm'): [Ln + Jn.derivative()/2, Jn, 1/3]}, conformal_vector=Ln, horizontal_grading={'Gp': 1, 'Gm': -1})
MN = N2.induced_module(N2.zhu_algebra().module({'Ln': 1/6, 'Jn': 1/3}, horizontal_grades=[(1/3,)]))
print(MN.horizontal_grading_group(), MN.horizontal_grade(MN.highest_weight_vector()))
print([MN.weight_space(1/6 + d/2, horizontal_grade=(1/3 - 1,)).dimension() for d in range(9)])
print(MN.character(3, horizontal_grade=(1/3,)))

Vector space of dimension 1 over Rational Field (1/3)
[0, 1, 0, 3, 0, 9, 0, 22, 0]
q^(1/6) + 3*q^(7/6) + 9*q^(13/6) + O(q^3)


## 7. Infinite top levels: affine Verma modules and admissible levels

Zhu algebras inherit the grading of $V$ (`A.horizontal_grading_group()`, `x.horizontal_grade()`). For affine algebras, the height functional on the grading group splits the generators of $A(V) = U(\mathfrak g)$ into raising, Cartan and lowering ones, and `A.verma_module(λ)` is the Verma module of $\mathfrak g$ of highest weight $\lambda$: infinite-dimensional, but graded with finite-dimensional pieces.

The example is $\mathfrak{sl}_2$ at the admissible level $k = -\frac12$ [KW1988] and the admissible weight $\lambda = -\frac12$. The Zhu algebra is an associative algebra, so `*` multiplies its elements and lets them act on its modules. The cell below prints the grading group, a grade and homogeneous components, then a vector $F^2 v$ of the Verma module, the action of $E$ on it, its grade, and the action of the image of the Sugawara vector, $\lambda(\lambda + 2)/(4(k + 2)) = -\frac18$.

In [27]:
Va = UniversalAffineVertexAlgebra(QQ, ['A', 1], level=-1/2, names=['E', 'H', 'F'], conformal_vector='sugawara')
Aa = Va.zhu_algebra(); Ez, Hz, Fz = Aa.gens()
print(Aa.horizontal_grading_group(), (Ez*Fz).horizontal_grade(), Aa.homogeneous_components(Ez*Ez + Ez))
Ua = Aa.verma_module({'H': -1/2}); v = Ua.highest_weight_vector()
x = Fz * (Fz * v); print(Ua, '\n', x, '|', Ez * x, '|', Ua.horizontal_grade(x), '|', Aa(Va.conformal_vector()) * v)

Ambient free module of rank 1 over the principal ideal domain Integer Ring (0) {(2): [E]^2, (1): [E]}
Verma module of highest weight {'H': -1/2} over Zhu algebra of Universal affine vertex algebra of type ['A', 1] at level -1/2 over Rational Field 
 [F]^2 v | -3*[F] v | (-9/2) | -1/8*v


**Affine Verma modules.** Inducing from $M_{\mathfrak{sl}_2}(\lambda)$ gives the affine Verma module $M(\lambda)$. Its conformal weight spaces are infinite-dimensional, but those of fixed $\mathfrak{sl}_2$-weight are finite. Submodules and the maximal submodule are computed piece by piece in a *twisted degree*, depth $+\,\mu(\text{grade})$, with $\mu$ a small multiple of minus the height, so that the highest weight vector is the only vector of degree 0 and every piece is finite. Only the pieces a question needs are computed, each a small linear algebra problem.

The simple quotients $L(-\frac12)$ and $L(-\frac32)$ are the two halves of the Ramond sector of the $\beta\gamma$ system: integer modes, with $\beta_0 v = 0$ [Rid2009]. The function `fock(d, c)` counts the states of the Ramond Fock space at depth $d$ and charge $c = \#\beta - \#\gamma$. The cell below compares, for both weights, the dimensions of the simple quotient by depth and $\mathfrak{sl}_2$-weight with these counts, and shows that a conformal weight space without a grade is refused.

In [28]:
def fock(d, c):
    return sum(1 for db in range(d + 1) for pb in Partitions(db) for pg in Partitions(d - db)
               if len(pb) - len(pg) - c >= 0)

for lam in (-1/2, -3/2):
    Ma = Va.induced_module(Aa.verma_module({'H': lam}))
    La = Ma.simple_quotient(); h = Ma.top_weight()
    ours = [[La.weight_space(h + d, horizontal_grade=(lam + m,)).dimension() for m in range(-6, 3, 2)] for d in range(4)]
    want = [[fock(d, lam + m + 1/2) for m in range(-6, 3, 2)] for d in range(4)]
    print(lam, h, ours == want, ours)
try:
    Ma.weight_space(h)
except ValueError as err:
    print('ValueError:', err)

-1/2 -1/8 True [[1, 1, 1, 1, 0], [2, 2, 2, 1, 0], [5, 5, 5, 3, 1], [10, 10, 9, 6, 2]]


-3/2 -1/8 True [[1, 1, 1, 1, 0], [2, 2, 2, 2, 1], [5, 5, 5, 4, 2], [10, 10, 10, 8, 4]]
ValueError: the conformal weight spaces of Induced module of Verma module of highest weight {'H': -3/2} over Zhu algebra of Universal affine vertex algebra of type ['A', 1] at level -1/2 over Rational Field over Universal affine vertex algebra of type ['A', 1] at level -1/2 over Rational Field are infinite dimensional: give a grade


**$\mathfrak{sl}_3$ at the admissible level $-\frac32$**, with $\lambda = (-\frac12, -\frac12)$ in fundamental weight coordinates. Of the reflections of the top, only the one by the highest root $\theta$ gives a singular vector of $M_{\mathfrak{sl}_3}(\lambda)$, since $\langle \lambda + \rho, \theta^\vee \rangle = 1$; so the top of $L(\lambda)$ loses one vector at $\lambda - \theta$, and the dimensions drop further at larger depths. The cell below prints, for depths $0, 1, 2$ and the weights $\lambda - a\alpha_1 - b\alpha_2$, the dimensions of $M(\lambda)$ and $L(\lambda)$.

In [29]:
Vs = UniversalAffineVertexAlgebra(QQ, ['A', 2], level=-3/2, conformal_vector='sugawara')
Ms = Vs.induced_module(Vs.zhu_algebra().verma_module({'alphacheck[1]': -1/2, 'alphacheck[2]': -1/2}))
Ls = Ms.simple_quotient(); hs = Ms.top_weight()
C = matrix(ZZ, [[2, -1], [-1, 2]]); lam = vector(QQ, [-1/2, -1/2])
for d in range(3):
    print(d, [(a, b, Ms.weight_space(hs + d, horizontal_grade=tuple(lam - C*vector([a, b]))).dimension(),
               Ls.weight_space(hs + d, horizontal_grade=tuple(lam - C*vector([a, b]))).dimension())
              for a in range(3) for b in range(3)])

0 [(0, 0, 1, 1), (0, 1, 1, 1), (0, 2, 1, 1), (1, 0, 1, 1), (1, 1, 2, 1), (1, 2, 2, 1), (2, 0, 1, 1), (2, 1, 2, 1), (2, 2, 3, 1)]


1 [(0, 0, 6, 3), (0, 1, 8, 3), (0, 2, 8, 3), (1, 0, 8, 3), (1, 1, 14, 4), (1, 2, 16, 4), (2, 0, 8, 3), (2, 1, 16, 4), (2, 2, 22, 4)]


2 [(0, 0, 33, 9), (0, 1, 44, 10), (0, 2, 47, 10), (1, 0, 44, 10), (1, 1, 72, 13), (1, 2, 85, 13), (2, 0, 47, 10), (2, 1, 85, 13), (2, 2, 116, 14)]


## 8. Modules over quotients

Let $Q = V/I$ be a quotient of a universal vertex algebra (or a presentation $F/R$ of a subalgebra, `S.presentation()`, see `06_subalgebras_ideals_quotients.ipynb`, section 6), and $U$ a finite-dimensional module over a Zhu algebra of $Q$ (`Q.zhu_algebra_up_to(W)`). Then `Q.induced_module_up_to(U, max_weight, max_depth)` is Dong–Li–Mason's universal module $\bar M_Q(U)$ with top level $U$ [DLM1998]: every positive-energy $Q$-module generated by $U$, simple or logarithmic, is a quotient of it.

It is computed as $\bar M_V(U)$ modulo the subspace generated by the $r_{(n)} u$, $r \in I$, $u \in U$ (and by the defects of the commutator formula of the generators, which occur when $F$ does not satisfy the Jacobi identity), in the depths $\le$ `max_depth`, from the relations of weight $\le$ `max_weight`. It maps onto $\bar M_Q(U)$ in these depths, and equals it once it no longer changes as `max_weight` and `max_depth` grow.

**Lee–Yang.** The Lee–Yang model is rational [Wan1993], so $\bar M_Q(U)$ is the simple module $L(U)$. From its two tops the cell below finds the two Rogers–Ramanujan characters of section 5, now as modules over the quotient. A top that is not a module over the Zhu algebra of $Q$ is reached by the relations, and refused.

In [30]:
QLY = VirLY.quotient([Lq.no(Lq) - 3/10*Lq.derivative(2)])
ZLY = QLY.zhu_algebra_up_to(4)
for h in (0, -1/5):
    M = QLY.induced_module_up_to(ZLY.module({'Lq': h}), 4, 10)
    print(h, [M.weight_space(h + d).dimension() for d in range(11)])
try:
    QLY.induced_module_up_to(VirLY.zhu_algebra().module({'Lq': 1/3}), 4, 3)
except ValueError as err:
    print('ValueError:', str(err)[:80], '...')

0 [1, 0, 1, 1, 1, 1, 2, 2, 3, 3, 4]


-1/5 [1, 1, 1, 1, 2, 2, 3, 3, 4, 5, 6]
ValueError: the relations of weight <= 4 reach the top level: Module of dimension 1 over Zhu ...


The triplet algebra $\mathcal W(2)$, a presentation of a subalgebra of symplectic fermions with logarithmic modules, is worked out in `15_triplet_modules.ipynb`: its Zhu algebra and simple modules, the module $\mathcal W(0) \to 2\mathcal W(1)$, and the projective cover of its vacuum module.

## 9. Morphisms of vertex algebras

A morphism $\varphi: V \to W$ of vertex algebras is a linear map with $\varphi(a_{(n)} b) = \varphi(a)_{(n)} \varphi(b)$ for all $n$ and $\varphi(|0\rangle) = |0\rangle$. It is determined by the images of generators, and `V.hom(images, W)` checks that the images satisfy the relations of $V$:

* for a universal vertex algebra, the OPEs of its strong generators;
* for a lattice vertex algebra $V_L$, conditions on the images of the $e^{\pm x_i}$ alone [LL2004, §6.5];
* for a quotient $V/I$, in addition that the given generators of $I$ map to zero.

These checks are exact, and then `checked_up_to_weight()` is `None`. `Hom(V, W)` is the set of morphisms, `φ(x)` applies a morphism, and `ψ * φ` composes two.

**The Frenkel–Kac construction** [FK1980] realises affine $\mathfrak{sl}_2$ at level 1 in the lattice vertex algebra $V_{A_1}$ [Kac1998] of section 2: $E \mapsto e^{\alpha}$, $H \mapsto J$, $F \mapsto e^{-\alpha}$. The cell below builds the morphism from $V^1(\mathfrak{sl}_2)$ (the algebra `VA` of section 3), checks that it is conformal (the Sugawara vector maps to $\frac14{:}JJ{:}$), and shows the images of ${:}EE{:}$ (zero) and ${:}EF{:}$ ($\frac12({:}JJ{:} + \partial J)$). With the sign of the image of $F$ changed, the first-order pole of $E$ and $F$ comes out wrong, and the map is refused.

In [31]:
Jl, ep, em = V.gens()                   # V = V_{A_1} with T = :JJ:/4 (section 2)
Ea, Ha, Fa = VA.gens()                  # VA = V^1(sl_2) (section 3)
fk = VA.hom([ep, Jl, em], V)
print(fk)
print(fk.parent() is Hom(VA, V), fk.checked_up_to_weight(), fk.is_conformal())
print(fk(Ea.no(Ea)), '|', fk(Ea.no(Fa)))
try:
    VA.hom([ep, Jl, -em], V)
except ValueError as err:
    print('ValueError:', err)

Vertex algebra morphism:
  From: Universal affine vertex algebra of type ['A', 1] at level 1 over Rational Field
  To:   Lattice vertex algebra of rank 1 with Gram matrix [2] over Rational Field
  Defn: E |--> e^(1)
        H |--> J0
        F |--> e^(-1)
True None True
0 | 1/2*:J0 J0: + 1/2*dJ0
ValueError: the images do not satisfy the OPE of E and F: pole 0 is -J0 instead of J0


**Kernel and image.** The kernel of a morphism is an ideal. `φ.kernel()` computes it weight space by weight space (for morphisms out of universal, lattice and tensor product vertex algebras that preserve conformal weights). For the Frenkel–Kac map it contains the ideal generated by ${:}EE{:}$, since $\varphi({:}EE{:}) = 0$, and the cell below shows that the two have the same dimensions in every weight computed: $\varphi$ factors through $L_1(\mathfrak{sl}_2) = V^1(\mathfrak{sl}_2)/\langle {:}EE{:} \rangle$ (see `06_subalgebras_ideals_quotients.ipynb`, section 4).

`φ.image()` is the subalgebra of $W$ generated by the images of the generators. Here its weight spaces are those of $V_{A_1}$ up to weight 4, so the map is onto in these weights (`is_surjective_up_to(4)`); it is injective only up to weight 1, since ${:}EE{:}$ has weight 2.

In [32]:
Kfk = fk.kernel(); Iee = VA.ideal([Ea.no(Ea)])
print(Kfk)
print([Kfk.weight_space(w).dimension() for w in range(5)], [Iee.weight_space(w).dimension() for w in range(5)])
Sfk = fk.image()
print([Sfk.weight_space(w).dimension() for w in range(5)], [V.weight_space(w).dimension() for w in range(5)])
print(fk.is_surjective_up_to(4), fk.is_injective_up_to(1), fk.is_injective_up_to(2))

Kernel of a map on Universal affine vertex algebra of type ['A', 1] at level 1 over Rational Field
[0, 0, 5, 15, 38] [0, 0, 5, 15, 38]
[1, 3, 4, 7, 13] [1, 3, 4, 7, 13]
True True False


**Composition.** The Chevalley involution $\sigma$ of $V^1(\mathfrak{sl}_2)$, $E \mapsto -F$, $H \mapsto -H$, $F \mapsto -E$, and the automorphism $\tau$ of $V_{A_1}$ with $e^{\pm\alpha} \mapsto -e^{\mp\alpha}$ (given by these two images only; it sends $J$ to $-J$) satisfy $\sigma^2 = 1$ and $\varphi \circ \sigma = \tau \circ \varphi$:

In [33]:
sigma = VA.hom([-Fa, -Ha, -Ea], VA)
tau = V.hom([-em, -ep], V)
print(tau(Jl), sigma * sigma == End(VA).identity(), fk * sigma == tau * fk)

-J0 True True


## 10. Module morphisms

A morphism $f: M \to N$ of $V$-modules is a linear map with $f(a_{(n)} m) = a_{(n)} f(m)$ for all $a \in V$ and $n \in \mathbb Z$. `M.hom(images, N)` gives it by the images of generators of $M$ and checks them:

* for an induced module $\bar M(U)$, the images of a basis of the top space $U$ must be killed by the positive modes of the strong generators, and the zero modes must act on them as on $U$ (the induction adjunction of [Zhu1996]);
* for the adjoint module, the image $w$ of the vacuum must be *vacuum-like*: $g_{(n)} w = 0$ for $n \ge 0$ and all generators $g$ [Li1994].

`Hom(M, N)` is a vector space, and `Hom(M, N).basis()` solves these conditions as linear equations on the images.

**A Verma module embedding.** The singular vector $s$ of $M(\frac12, \frac12)$ of section 5 gives a morphism from the Verma module $M(\frac12, \frac52)$. `basis()` finds it, with another normalisation; $L_{-2} v$ is not singular, and is refused as an image.

In [34]:
M = Ising.verma_module(1/2); v = M.highest_weight_vector()
M52 = Ising.verma_module(5/2)
f, = Hom(M52, M).basis()
print(f)
s = Ti.mode(v, -2) - 3/4 * Ti.mode(Ti.mode(v, -1), -1)
print(M52.hom([s], M) == -3/4 * f)
try:
    M52.hom([Ti.mode(v, -2)], M)
except ValueError as err:
    print('ValueError:', err)

Vertex algebra module morphism:
  From: Verma module of highest weight {'Ti': 5/2} over Universal vertex algebra on generators ['Ti'] over Rational Field
  To:   Verma module of highest weight {'Ti': 1/2} over Universal vertex algebra on generators ['Ti'] over Rational Field
  Defn: v |--> -4/3*Ti_{-2} v + Ti_{-1} Ti_{-1} v
True
ValueError: the image of v is not killed by the mode Ti_(2) (it is not in Ω(N))


**Kernels, images and cokernels** of module morphisms are modules: `f.kernel()` is a submodule of the domain, `f.image()` the submodule of the codomain generated by the images, and `f.cokernel()` the quotient by the image. The map $f: M(\frac12, \frac52) \to M(\frac12, \frac12)$ is injective up to depth 6, and its image is the submodule $N$ generated by $s$ in section 5. The projection $p$ onto the simple quotient $L(\frac12, \frac12)$ (`M.simple_quotient().cover()`) has the maximal submodule as its kernel (compare the dimensions in section 5), and $p \circ f = 0$:

In [35]:
print(f.is_injective_up_to(6), [f.image().weight_space(1/2 + d).dimension() for d in range(9)])
print([f.cokernel().weight_space(1/2 + d).dimension() for d in range(9)])
p = M.simple_quotient().cover()
print([p.kernel().weight_space(1/2 + d).dimension() for d in range(9)], (p * f).is_zero())

True [0, 0, 1, 1, 2, 3, 5, 7, 11]
[1, 1, 1, 2, 3, 4, 6, 8, 11]
[0, 0, 1, 2, 3, 5, 8, 11, 17] True


**Vacuum-like vectors and Schur's lemma.** The adjoint module maps to a module $N$ by $|0\rangle \mapsto w$ exactly when $w$ is vacuum-like. In the Verma module $M(\frac12, 0)$ no nonzero vector of weight $0$ is vacuum-like (there $L_{-1} v \neq 0$); in the corresponding simple quotient, the vector $v$ is vacuum-like, and the map is the projection of the vacuum module. The morphisms between the lattice modules of section 2 follow Schur's lemma, and the endomorphisms of $V_{1/2+L} \oplus V_{1/2+L}$ are the $2 \times 2$ matrices:

In [36]:
Ad = Ising.adjoint_module()
print(Hom(Ad, Ising.verma_module(0)).dimension(), Hom(Ad, Ising.verma_module(0).simple_quotient()).basis()[0].im_gens())
M0, M1 = V.simple_modules()
print([[Hom(X, Y).dimension() for Y in (M0, M1)] for X in (M0, M1)], End(M1.direct_sum(M1)).dimension())

0 (v,)
[[1, 0], [0, 1]] 4


## 11. Pullbacks, twists and spectral flow

**Pullbacks.** For a morphism $\varphi: U \to V$ of vertex algebras and a $V$-module $M$, the pullback $\varphi^* M$ (`M.pullback(phi)`) is the space $M$ with $a_{(n)} m = \varphi(a)_{(n)} m$ for $a \in U$. It is conformally graded, with the weights of $M$, when $\varphi$ is conformal. Restricting a module to a subalgebra is the pullback along the embedding.

The Virasoro algebra at $c = 1$ maps to $V^1(\mathfrak{sl}_2)$ by $L \mapsto T_{\mathrm{sug}}$. The pullback of the simple module `L1` with the doublet as its top (section 5) is a Virasoro module with the same character. Both vectors of the top are Virasoro highest weight vectors of weight $\frac14$, so `Hom` from the Virasoro Verma module $M(1, \frac14)$ has dimension 2:

In [37]:
Vir1 = va.Virasoro(QQ, 1, conformal_vector='L')
phi = Vir1.hom([VA.conformal_vector()], VA)
L1 = VA.induced_module(AA.module(doublet)).simple_quotient()
P = L1.pullback(phi)
print(P.category())
print(P.character(4) == L1.character(4), P.character(4))
print([g.im_gens() for g in Hom(Vir1.verma_module(1/4), P).basis()])

Category of conformally graded modules with basis over Universal vertex algebra on generators ['L'] over Rational Field
True 2*q^(1/4) + 2*q^(5/4) + 6*q^(9/4) + 8*q^(13/4) + O(q^4)
[(u[0],), (u[1],)]


**Twists.** For an automorphism $\sigma$ of $V$ the pullback is the twist $M^\sigma$ (`M.twist(sigma)`). The Chevalley involution $\sigma$ of section 9 changes the sign of $H_0$ on the top of the Weyl module of the doublet: $u_0$ now has $H_0$-eigenvalue $-1$ and $u_1$ eigenvalue $1$. The twisted module is again a Weyl module of the doublet: `Hom` finds the map $u_0 \mapsto -u_1$, $u_1 \mapsto u_0$, and it is bijective up to depth 3 (`is_isomorphism_up_to`).

In [38]:
W2 = VA.induced_module(AA.module(doublet))
W2s = W2.twist(sigma)
u0, u1 = [W2s(W2.monomial(((), j))) for j in (0, 1)]
print(Ha.mode(u0, 0), '|', Ha.mode(u1, 0), '|', W2s.character(3) == W2.character(3))
iso, = Hom(W2, W2s).basis()
print(iso.im_gens(), iso.is_isomorphism_up_to(3))

-u[0] | u[1] | True
(-u[1], u[0]) True


**Spectral flow** is H. Li's $\Delta$-operator [Li1997]. For $h \in V$ with $h_{(n)} h = \delta_{n,1}\, \gamma\, |0\rangle$ ($n \ge 0$), and $h_{(0)}$ acting on $V$ with integer eigenvalues,
$$\Delta(h, z) = z^{h_{(0)}} \exp\Big(\sum_{k \ge 1} \frac{h_{(k)}}{-k} (-z)^{-k}\Big),$$
and $Y_M(\Delta(h, z) a, z)$ is a new module structure on the space $M$ (`M.spectral_flow(h)`). For $h$ primary of weight 1 the new $L_0$ is $L_0 + h_{(0)} + \gamma/2$, so the weights shift with the charge.

**Heisenberg Fock modules.** The Heisenberg algebra $\alpha(z)\alpha(w) \sim (z-w)^{-2}$ with $T = \frac12 {:}\alpha\alpha{:}$ has the Fock modules $F_p$ ($\alpha_0 = p$, top weight $p^2/2$), induced from the modules of its Zhu algebra. The flow of $F_0$ by $3\alpha$ ($\gamma = 9$) gives $\alpha_0 = 3$ and top weight $\frac92$ on the old vacuum, and it is isomorphic to $F_3$: the map $v \mapsto v$ is bijective up to depth 5.

In [39]:
Hb = va.FreeBosons(QQ); al = Hb.gen(0)
Hb = Hb.with_conformal_vector(al.no(al) / 2); al = Hb.gen(0)
fock = lambda p: Hb.induced_module(Hb.zhu_algebra().module({'alpha': p}))
F0, F3 = fock(0), fock(3)
S3 = F0.spectral_flow(3*al)
w = S3(F0.highest_weight_vector())
print(al.mode(w, 0), '|', S3.character(8), '|', F3.character(8))
iso, = Hom(F3, S3).basis()
print(iso.im_gens(), iso.is_isomorphism_up_to(5))

3*v | q^(9/2) + q^(11/2) + 2*q^(13/2) + 3*q^(15/2) + O(q^8) | q^(9/2) + q^(11/2) + 2*q^(13/2) + 3*q^(15/2) + O(q^8)
(v,) True


**Affine $\mathfrak{sl}_2$ at level 1.** The flow of the simple vacuum module `L0` by $H/2$ ($\gamma = k/2 = \frac12$) puts the old vacuum $v$ at weight $\frac14$, with $H_0 v = v$, $E_0 v = 0$ and $F_0 v$ the old $F_{-1} v$: a highest weight vector of the doublet. The flowed module is isomorphic to `L1` [Li1997, Prop. 3.5]. The flow keeps the grades of the vectors of `L0`, so the isomorphism lowers grades by 1: `horizontal_grade=G([0])` asks for the image of $u_0$ (of grade 1 in `L1`) to have grade 0. `is_isomorphism_up_to(3, g)` compares the graded pieces of depth $\le 3$.

In [40]:
L0 = VA.induced_module(AA.module({'E': 0, 'H': 0, 'F': 0})).simple_quotient()
Sf = L0.spectral_flow(Ha/2)
vf = Sf(L0.highest_weight_vector())
print(vf.degree(), '|', Ha.mode(vf, 0), '|', Ea.mode(vf, 0), '|', Fa.mode(vf, 0))
G = L1.horizontal_grading_group()
g, = Hom(L1, Sf).basis(horizontal_grade=G([0]))
print(g.im_gens(), all(g.is_isomorphism_up_to(3, G([n])) for n in (-3, -1, 1, 3)))

1/4 | v | 0 | F_{-1} v


(v, F_{-1} v) True


## 12. Limitations

* **Infinite top spaces.** Relaxed modules (dense modules of the Zhu algebra as top spaces, infinite in both directions) are not available: their submodules need more than the twisted degree of section 7. Verma modules of quotients of Zhu algebras, and infinite top spaces over quotients $V/I$, are not available either.
* **Up to a depth.** Modules over quotients (section 8) are computed from the relations up to `max_weight`, in the depths up to `max_depth`; the code cannot tell when the result has stabilised. Statements such as `is_isomorphism_up_to(n)` and `is_injective_up_to(n)` say nothing about larger depths.
* **Missing constructions.** Quotients of submodules; tensor products of modules (but see fusion products, `11_fusion_and_extensions.ipynb`); contragredient modules. Restriction to a subalgebra or coset is only available as a pullback along a morphism into $V$ (section 11).
* **Hom spaces** out of submodules are not computed; kernels of vertex algebra morphisms that do not preserve conformal weights are not computed.
* **Twisted modules.** There are no Ramond-twisted (or other twisted) induced modules: the H-twisted Zhu algebra gives the top levels (section 3), but they are not induced to modules. Generators of weight 0 are not supported for induced modules.
* **Speed.** The module actions of lattice modules run in Python, not in the C++ engines, so they are slower than products in $V$.

## References

- [Kac1998] V. G. Kac, *Vertex algebras for beginners*, 2nd ed., Univ. Lecture Ser. 10, AMS (1998), [doi:10.1090/ulect/010](https://doi.org/10.1090/ulect/010). — modules, lattice vertex algebras and their modules (sections 1, 2, 9).
- [LL2004] J. Lepowsky, H. Li, *Introduction to vertex operator algebras and their representations*, Progr. Math. 227, Birkhäuser (2004), [doi:10.1007/978-0-8176-8186-9](https://doi.org/10.1007/978-0-8176-8186-9). — modules (section 1), morphisms out of lattice vertex algebras (§6.5, section 9).
- [Zhu1996] Y. Zhu, *Modular invariance of characters of vertex operator algebras*, J. Amer. Math. Soc. 9 (1996), 237–302, [doi:10.1090/S0894-0347-96-00182-8](https://doi.org/10.1090/S0894-0347-96-00182-8). — Zhu's correspondence, induced modules and their simple quotients $L(U)$ (sections 3, 4, 10).
- [FZ1992] I. B. Frenkel, Y. Zhu, *Vertex operator algebras associated to representations of affine and Virasoro algebras*, Duke Math. J. 66 (1992), 123–168, [doi:10.1215/S0012-7094-92-06604-X](https://doi.org/10.1215/S0012-7094-92-06604-X). — Zhu algebras of affine and Virasoro vertex algebras (section 3).
- [DLM1998] C. Dong, H. Li, G. Mason, *Vertex operator algebras and associative algebras*, J. Algebra 206 (1998), 67–96, [arXiv:q-alg/9612010](https://arxiv.org/abs/q-alg/9612010). — the universal modules $\bar M_Q(U)$ of quotients (section 8).
- [Don1993] C. Dong, *Vertex algebras associated with even lattices*, J. Algebra 161 (1993), 245–265, [doi:10.1006/jabr.1993.1217](https://doi.org/10.1006/jabr.1993.1217). — the modules of lattice vertex algebras (section 2).
- [DLM1997] C. Dong, H. Li, G. Mason, *Regularity of rational vertex operator algebras*, Adv. Math. 132 (1997), 148–166, [arXiv:q-alg/9508018](https://arxiv.org/abs/q-alg/9508018). — the modules $V_{\lambda+L}$ of lattice vertex algebras (section 2).
- [Kac1979] V. G. Kac, *Contravariant form for infinite-dimensional Lie algebras and superalgebras*, in: Group theoretical methods in physics, Lecture Notes in Phys. 94, Springer (1979), 441–445. — the Kac determinant (section 4).
- [BPZ1984] A. A. Belavin, A. M. Polyakov, A. B. Zamolodchikov, *Infinite conformal symmetry in two-dimensional quantum field theory*, Nuclear Phys. B 241 (1984), 333–380, [doi:10.1016/0550-3213(84)90052-X](https://doi.org/10.1016/0550-3213(84)90052-X). — the Ising and Lee–Yang models (sections 4, 5).
- [Zam1985] A. B. Zamolodchikov, *Infinite additional symmetries in two-dimensional conformal quantum field theory*, Theoret. and Math. Phys. 65 (1985), 1205–1213, [doi:10.1007/BF01036128](https://doi.org/10.1007/BF01036128). — the $W_3$ algebra (section 4).
- [Wan1993] W. Wang, *Rationality of Virasoro vertex operator algebras*, Int. Math. Res. Not. 1993, no. 7, 197–211, [doi:10.1155/S1073792893000212](https://doi.org/10.1155/S1073792893000212). — the minimal models are rational (section 8).
- [KW1988] V. G. Kac, M. Wakimoto, *Modular invariant representations of infinite-dimensional Lie algebras and superalgebras*, Proc. Nat. Acad. Sci. U.S.A. 85 (1988), 4956–4960, [doi:10.1073/pnas.85.14.4956](https://doi.org/10.1073/pnas.85.14.4956). — admissible levels and weights (section 7).
- [Rid2009] D. Ridout, *$\widehat{\mathfrak{sl}}(2)_{-1/2}$: a case study*, Nuclear Phys. B 814 (2009), 485–521, [arXiv:0810.3532](https://arxiv.org/abs/0810.3532). — affine $\mathfrak{sl}_2$ at level $-\frac12$ and the $\beta\gamma$ system (section 7).
- [FK1980] I. B. Frenkel, V. G. Kac, *Basic representations of affine Lie algebras and dual resonance models*, Invent. Math. 62 (1980), 23–66, [doi:10.1007/BF01391662](https://doi.org/10.1007/BF01391662). — affine $\mathfrak{sl}_2$ at level 1 from the $A_1$ lattice (section 9).
- [Li1994] H. Li, *Symmetric invariant bilinear forms on vertex operator algebras*, J. Pure Appl. Algebra 96 (1994), 279–297, [doi:10.1016/0022-4049(94)90104-X](https://doi.org/10.1016/0022-4049(94)90104-X). — vacuum-like vectors, morphisms out of the adjoint module (section 10).
- [Li1997] H. Li, *The physics superselection principle in vertex operator algebra theory*, J. Algebra 196 (1997), 436–457, [doi:10.1006/jabr.1997.7126](https://doi.org/10.1006/jabr.1997.7126). — spectral flow (section 11).